In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, mean_squared_error, mean_absolute_error, r2_score,
)
from sklearn.impute import SimpleImputer
from scipy.stats import chi2_contingency
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')

: 

In [ ]:
df_train = pd.read_csv("CC_education_economy_train.csv")
df_test = pd.read_csv("CC_education_economy_test.csv")
df_kaggle = pd.read_csv("CC_private_test.csv")

submission_ids = df_kaggle['id']
df_kaggle = df_kaggle.drop(columns='id')

In [ ]:
print('train data:', df_train.shape)

print(df_train.head(3))

In [ ]:
print(df_kaggle.info())

# 3.1. Exploratory Data Analysis

## 3.1.0 Missing Data

In [ ]:
sns.heatmap(df_train.isnull(), yticklabels = False, cbar = False
            , cmap = 'tab20c_r')
plt.title('Missing Data: Training Set')
plt.show()

## 3.1.1 Analysis of attribute types and their value ranges.


In [ ]:
""" This is where I get all of the continous attributes stats"""
def extract_cols(df_train: pd.DataFrame, cls: type, column_dropped: str | None = None) -> list[str]:
    if column_dropped is not None:
        cols = df_train.select_dtypes(include=cls).drop(columns=column_dropped).columns.to_list()
    else:
        cols = df_train.select_dtypes(include=cls).columns.to_list()

    return cols

continous_cols = extract_cols(df_train, ['int64'], 'salary')

stats_continous = df_train[continous_cols].describe()

print("Statistici pentru atribute continue (int64):")
display(stats_continous)

sns.boxplot(data = df_train[continous_cols], palette= 'GnBu_d').set_title('Continous Attributes')
plt.show()

In [ ]:
""" I decided to analyze further the skills_count attribute"""

df_train.boxplot(column = 'skills_count')

In [ ]:
""" Here I create a tabel full of discrete attributes """
def extract_non_numeric_cols(df_train: pd.DataFrame, cols: list[str]):
    rezultate = []

    for col in cols:
        rezultate.append({
            'Atribut': col,
            'Count (non-missing)': df_train[col].count(),
            'Unique Values': len(df_train[col].unique())
        })

    final_table = pd.DataFrame(rezultate)

    return final_table

discret_cols = extract_cols(df_train, ['object'], 'vacation')
final_discret_table = extract_non_numeric_cols(df_train, discret_cols)

display(final_discret_table)

In [ ]:
"""   I decided to take job_title into further analyzation
    because it has the biggest number of unique values and
    remote_work because of it's lowest number of non-missing
    values."""

df_train['job_title'].hist(width=0.6)

plt.title("Histogram: Frequency of Job titles")
plt.xlabel("Job Titles")
plt.ylabel("Frequency")

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

df_train['industry'].hist(width=0.6)

plt.title("Histogram: Frequency of Industry")
plt.xlabel("Industry")
plt.ylabel("Frequency")

plt.xticks(rotation=45, ha='right')
plt.tight_layout()

plt.show()

In [ ]:
""" Here I create a tabel full of ordinal attributes """

ordinal_cols = extract_cols(df_train, ['float64'])
final_categorial_table = extract_non_numeric_cols(df_train, ordinal_cols)

display(final_categorial_table)

## 3.1.2 Class balance analysis.

In [ ]:
""" I use countplot for representation of classes. """

sns.countplot(x='vacation', data=df_train, palette='viridis')

plt.title("Analiza echilibrului de clase (Vacation)")
plt.xlabel("Categorie Vacanță")
plt.ylabel("Număr de exemple")
plt.show()

In [ ]:
""" Trying to show the values and procentages of all classes """
print("----- DATA INFO -----")
print("UNIQUE VALUES OF CLASSES")
print(df_train['vacation'].value_counts())

print("\nPROCENTAGE FOR EVERY CLASS")
print(df_train['vacation'].value_counts(normalize=True) * 100)
print("----- DATA INFO -----")

## 3.1.3 Correlation analysis between attributes.

In [ ]:
"""Here I did analyze the correlation between all of the continous numerical atributes"""

nr_of_cols = len(continous_cols)
correlation_matrix = df_train[continous_cols].corr()

fig = plt.figure(figsize=(10, 10))

ax = fig.add_subplot(111)
cax = ax.matshow(correlation_matrix, vmin=-1, vmax=1)

for i in range(nr_of_cols):
    for j in range(nr_of_cols):
        ax.text(j, i, f'{correlation_matrix.iloc[i, j]:.2f}',
                ha='center', va='center', fontsize=8)

fig.colorbar(cax)
ticks = np.arange(0,nr_of_cols,1)

ax.set_xticks(ticks)
ax.set_yticks(ticks)

ax.set_xticklabels(continous_cols)
ax.set_yticklabels(continous_cols)

ax.set_title("Correlation Matrix of Numerical Attributes", pad = 20)

plt.show()

In [ ]:
"""Here I did analyze the correlation between all of the categorial atributes"""

def correlation_non_numeric_values(df_train: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    rezultate = []

    for col_i in cols:
        for col_j in cols:
            if col_i == col_j:
                continue

            CrosstabResult = pd.crosstab(index=df_train[col_i], columns=df_train[col_j])
            ChiSqResult = chi2_contingency(CrosstabResult)

            rezultate.append({
                "Corelation": f"between {col_i} and {col_j}",
                "Chi-sq test result": f"The result is {ChiSqResult[1]}"
            })

    corr_tabel = pd.DataFrame(rezultate)

    return corr_tabel

categorial_corr_table = correlation_non_numeric_values(df_train, discret_cols)
display(categorial_corr_table)

In [ ]:
# numeric vs classification task
sns.boxplot(x="vacation", y="skills_count", data=df_train)
plt.title("Skills vs Vacation")
plt.show()

# categorial vs classification task
sns.countplot(x="education_level", hue="vacation", data=df_train)
plt.title("Education vs Vacation")
plt.xticks(rotation=45)
plt.show()

# numeric vs regression task
sns.scatterplot(x="skills_count", y = "salary", data=df_train)
plt.title("Skills vs Salary")
plt.show()

# categorial vs regression task
sns.boxplot(x="education_level", y="salary", data=df_train)
plt.title("Education vs Salary")
plt.xticks(rotation=45)
plt.show()

# 3.2. Data Preprocessing

## 3.2.1 Handling missing data.

In [ ]:
"""     Here I decided that for the missing data to use a prediction algoritm
    which would generate the new values  """

def predict_imputation(df, feature, excluded_columns=None):
    df = df.copy()
    
    if excluded_columns is None:
        excluded_columns = []

    drop_columns = list(set(excluded_columns + [feature]))

    df_known = df[df[feature].notnull()]
    df_missing = df[df[feature].isnull()]

    X_train = df_known.drop(columns=drop_columns)
    y_train = df_known[feature]
    X_pred = df_missing.drop(columns=drop_columns)

    categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
    if len(categorical_cols) > 0:
        X_train_enc = pd.get_dummies(X_train, columns=categorical_cols, dummy_na=False)
        X_pred_enc = pd.get_dummies(X_pred, columns=categorical_cols, dummy_na=False)
        X_pred_enc = X_pred_enc.reindex(columns=X_train_enc.columns, fill_value=0)
    else:
        X_train_enc = X_train
        X_pred_enc = X_pred

    target_encoder = LabelEncoder()
    y_train = target_encoder.fit_transform(y_train)

    model = RandomForestClassifier(
            n_estimators=48,
            max_depth=58,
            min_samples_leaf=4,
            class_weight='balanced_subsample',
        )

    model.fit(X_train_enc, y_train)

    y_pred = model.predict(X_pred_enc)

    y_pred = target_encoder.inverse_transform(y_pred).flatten()

    df.loc[df[feature].isnull(), feature] = y_pred

    return df

df_simple = predict_imputation(df_train, "remote_work", excluded_columns=['vacation', 'salary'])
df_test = predict_imputation(df_test, "remote_work", excluded_columns=['vacation', 'salary'])
df_kaggle = predict_imputation(df_kaggle, "remote_work")

print("Noua distribuție (proporțională):")
print(df_simple["remote_work"].value_counts())
print(df_test["remote_work"].value_counts())
print(df_kaggle["remote_work"].value_counts())


## 3.2.2 Extreme values handling.

In [ ]:
"""I created this code cell to handle the extreme values of skills_count attribute"""
def mask_outliers(df: pd.DataFrame, column: str, Q1_value: float, Q3_value: float) -> pd.DataFrame:
    Q1 = df[column].quantile(Q1_value)
    Q3 = df[column].quantile(Q3_value)

    IQR = Q3 - Q1

    threshold = 1.5
    lower = Q1 - threshold * IQR
    upper = Q3 + threshold * IQR

    df[column] = df[column].mask(
        (df[column] < lower) | (df[column] > upper),
        np.nan
    )

    return df

outliers = 'skills_count'
df_simple = mask_outliers(df_simple, outliers, 0.25, 0.75)
df_test =  mask_outliers(df_test, outliers, 0.25, 0.75)
df_kaggle = mask_outliers(df_kaggle, outliers, 0.25, 0.75)

imp2 = SimpleImputer(missing_values=np.nan, strategy='mean')

df_simple[[outliers]] = imp2.fit_transform(df_simple[[outliers]])
df_test[[outliers]] = imp2.transform(df_test[[outliers]])
df_kaggle[[outliers]] = imp2.transform(df_kaggle[[outliers]])

display(df_simple[outliers].describe())
display(df_test[outliers].describe())
display(df_kaggle[outliers].describe())


## 3.2.3 Redundant attributes handling.

In [ ]:
"""After further analysis I decide to remove the total_days_worked attribute"""

df_kaggle = df_kaggle.drop(columns=['total_days_worked'])
df_simple = df_simple.drop(columns=['total_days_worked'])
df_test = df_test.drop(columns=['total_days_worked'])

continous_cols = [col for col in continous_cols if col != 'total_days_worked']

## 3.2.4 Data normalization and standardization.

In [ ]:
"""Here I apply the Standard Scaler transformation for normalization"""

scaler = StandardScaler()

df_simple[continous_cols] = scaler.fit_transform(df_simple[continous_cols])
df_test[continous_cols] = scaler.transform(df_test[continous_cols])
df_kaggle[continous_cols] = scaler.transform(df_kaggle[continous_cols])


# 3.3 Model Training and Evaluation

In [ ]:
def split_data(df: pd.DataFrame, predict_attributes: list[str]):
    X = df.drop(columns = predict_attributes)
    Y = df[predict_attributes[0]]

    X_train, X_val, Y_train, Y_val = train_test_split(
        X, Y,
        test_size=0.25,
        random_state=42,
        stratify=Y
    )

    return X_train, X_val, Y_train, Y_val

## Classification models training and evaluation.

In [ ]:
min_leaf_table = []

def experiment_tree(model, algoritm, dataset_details, X_train, Y_train, X_val, Y_val, table):
    model.fit(X_train, Y_train)

    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)

    acc_train = accuracy_score(Y_train, y_pred_train)
    acc_val = accuracy_score(Y_val, y_pred_val)
    prec_train = precision_score(Y_train, y_pred_train, average='weighted', zero_division=0)
    prec_val = precision_score(Y_val, y_pred_val, average='weighted', zero_division=0)
    rec_train = recall_score(Y_train, y_pred_train, average='weighted', zero_division=0)
    rec_val = recall_score(Y_val, y_pred_val, average='weighted', zero_division=0)
    f1_train = f1_score(Y_train, y_pred_train, average='weighted', zero_division=0)
    f1_val = f1_score(Y_val, y_pred_val, average='weighted', zero_division=0)

    table.append({
        "Algoritm": algoritm,
        "Details": dataset_details,
        "Acc_Train": acc_train,
        "Acc_Val": acc_val,
        "Prec_Train": prec_train,
        "Prec_Val": prec_val,
        "Rec_Train": rec_train,
        "Rec_Val": rec_val,
        "F1_Train": f1_train,
        "F1_Val": f1_val,
    })
    
    return model, y_pred_val

def ml_training_tree(df, dataset_details, table):
    X_train, X_val, Y_train, Y_val = split_data(df, ['vacation', 'salary'])

    le_local = LabelEncoder()
    Y_train_enc = le_local.fit_transform(Y_train)
    Y_val_enc = le_local.transform(Y_val)

    X_train_enc = pd.get_dummies(X_train, columns=discret_cols, dummy_na=False)
    X_val_enc = pd.get_dummies(X_val, columns=discret_cols, dummy_na=False)
    X_val_enc = X_val_enc.reindex(columns=X_train_enc.columns, fill_value=0)

    model_entropy = DecisionTreeClassifier(
        criterion='entropy',
        min_samples_leaf=48,
        max_depth=13,
    )
    experiment_tree(model_entropy, 'entropy', dataset_details, X_train_enc, Y_train_enc, X_val_enc, Y_val_enc, table)
    
    model_log_loss = DecisionTreeClassifier(
        criterion='log_loss',
        min_samples_leaf=48,
        max_depth=13,
    )
    experiment_tree(model_log_loss, 'log_loss', dataset_details, X_train_enc, Y_train_enc, X_val_enc, Y_val_enc, table)

    model_log_loss = DecisionTreeClassifier(
        criterion='log_loss',
        min_samples_leaf=100,
        max_depth=13,
    )

    experiment_tree(model_log_loss, 'entropy', dataset_details + ' 100 min_samples leaf', X_train_enc, Y_train_enc, X_val_enc, Y_val_enc, table)

    model_log_loss = DecisionTreeClassifier(
        criterion='log_loss',
        min_samples_leaf=120,
        max_depth=10,
    )

    experiment_tree(model_log_loss, 'log_loss', dataset_details + ' 120 min_samples leaf + 10 max_depth', X_train_enc, Y_train_enc, X_val_enc, Y_val_enc, table)

    model_log_loss = DecisionTreeClassifier(
        criterion='entropy',
        min_samples_leaf=150,
        max_depth=10,
    )

    experiment_tree(model_log_loss, 'entropy', dataset_details + ' 150 min_samples leaf + 12 max_depth', X_train_enc, Y_train_enc, X_val_enc, Y_val_enc, table)

In [ ]:
ml_training_tree(df_simple, 'simple df', min_leaf_table)

df_min_leaf = pd.DataFrame(min_leaf_table)

df_sortat = df_min_leaf.sort_values(by=["Acc_Val", "Prec_Val"], ascending=[False, False])
cea_mai_buna_linie = df_sortat.iloc[0]

print("Cea mai bună configurație găsită pentru Arbori de Decizie este:\n")
print(cea_mai_buna_linie)

In [ ]:
istoric_random_forest = []

def experiment_randomForest(model, dataset_details, X_train, Y_train, X_val, Y_val, table):
    model.fit(X_train, Y_train)

    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)

    acc_train = accuracy_score(Y_train, y_pred_train)
    acc_val = accuracy_score(Y_val, y_pred_val)
    prec_train = precision_score(Y_train, y_pred_train, average='weighted', zero_division=0)
    prec_val = precision_score(Y_val, y_pred_val, average='weighted', zero_division=0)
    rec_train = recall_score(Y_train, y_pred_train, average='weighted', zero_division=0)
    rec_val = recall_score(Y_val, y_pred_val, average='weighted', zero_division=0)
    f1_train = f1_score(Y_train, y_pred_train, average='weighted', zero_division=0)
    f1_val = f1_score(Y_val, y_pred_val, average='weighted', zero_division=0)

    table.append({
        "Model": f'{dataset_details}',
        "Acc_Train": acc_train,
        "Acc_Val": acc_val,
        "Prec_Train": prec_train,
        "Prec_Val": prec_val,
        "Rec_Train": rec_train,
        "Rec_Val": rec_val,
        "F1_Train": f1_train,
        "F1_Val": f1_val,
    })
    
    return model, y_pred_val

def ml_training_random(df, dataset_details, table):
    X_train, X_val, Y_train, Y_val = split_data(df, ['vacation', 'salary'])

    le_local = LabelEncoder()
    Y_train_enc = le_local.fit_transform(Y_train)
    Y_val_enc = le_local.transform(Y_val)

    X_train_enc = pd.get_dummies(X_train, columns=discret_cols, dummy_na=False)
    X_val_enc = pd.get_dummies(X_val, columns=discret_cols, dummy_na=False)
    X_val_enc = X_val_enc.reindex(columns=X_train_enc.columns, fill_value=0)

    model_rf = RandomForestClassifier(
        n_estimators=48,
        max_depth=58,
        min_samples_leaf=4,
    )
    
    experiment_randomForest(model_rf, dataset_details, X_train_enc, Y_train_enc, X_val_enc, Y_val_enc, table)

    model_rf = RandomForestClassifier(
        n_estimators=48,
        max_depth=15,
        min_samples_leaf=3,
        class_weight='balanced_subsample'
    )
    
    experiment_randomForest(model_rf, dataset_details + ' max_depth = 60', X_train_enc, Y_train_enc, X_val_enc, Y_val_enc, table)

    model_rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        min_samples_leaf=4,
        class_weight='balanced_subsample'
    )
    
    experiment_randomForest(model_rf, dataset_details + ' n_estimators = 100 + max_depth = 20', X_train_enc, Y_train_enc, X_val_enc, Y_val_enc, table)

ml_training_random(df_simple, 'Random', istoric_random_forest)

df_istoric = pd.DataFrame(istoric_random_forest)
df_istoric_sorted = df_istoric.sort_values(by=["Acc_Val", "F1_Val"], ascending=[False, False])

print("Cea mai bună configurație găsită pentru Random Forest este:\n")
print(df_istoric_sorted.iloc[0])

In [ ]:
istoric_histgb = []

def experiment_histGB(model, dataset_details, X_train, Y_train, X_val, Y_val, table):
    model.fit(X_train, Y_train)

    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)

    acc_train = accuracy_score(Y_train, y_pred_train)
    acc_val = accuracy_score(Y_val, y_pred_val)
    prec_train = precision_score(Y_train, y_pred_train, average='weighted', zero_division=0)
    prec_val = precision_score(Y_val, y_pred_val, average='weighted', zero_division=0)
    rec_train = recall_score(Y_train, y_pred_train, average='weighted', zero_division=0)
    rec_val = recall_score(Y_val, y_pred_val, average='weighted', zero_division=0)
    f1_train = f1_score(Y_train, y_pred_train, average='weighted', zero_division=0)
    f1_val = f1_score(Y_val, y_pred_val, average='weighted', zero_division=0)

    table.append({
        "Model": f'{dataset_details}',
        "Acc_Train": acc_train,
        "Acc_Val": acc_val,
        "Prec_Train": prec_train,
        "Prec_Val": prec_val,
        "Rec_Train": rec_train,
        "Rec_Val": rec_val,
        "F1_Train": f1_train,
        "F1_Val": f1_val,
    })
    
    return model, y_pred_val

def ml_training_histGB(df, dataset_details, table):
    X_train, X_val, Y_train, Y_val = split_data(df, ['vacation', 'salary'])

    le_local = LabelEncoder()
    Y_train_enc = le_local.fit_transform(Y_train)
    Y_val_enc = le_local.transform(Y_val)

    X_train_enc = pd.get_dummies(X_train, columns=discret_cols, dummy_na=False)
    X_val_enc = pd.get_dummies(X_val, columns=discret_cols, dummy_na=False)
    X_val_enc = X_val_enc.reindex(columns=X_train_enc.columns, fill_value=0)

    model_histgb = HistGradientBoostingClassifier(
        max_iter=400,
        max_depth=20,
        learning_rate=0.10,
    )
    
    experiment_histGB(model_histgb, dataset_details, X_train_enc, Y_train_enc, X_val_enc, Y_val_enc, table)

    model_histgb = HistGradientBoostingClassifier(
        max_iter=400,
        max_depth=15,
        learning_rate=0.10,
    )
    
    experiment_histGB(model_histgb, dataset_details + ' max_depth', X_train_enc, Y_train_enc, X_val_enc, Y_val_enc, table)

ml_training_histGB(df_simple, 'HistGB model', istoric_histgb)

df_istoric_histgb = pd.DataFrame(istoric_histgb)
df_istoric_histgb_sorted = df_istoric_histgb.sort_values(by=["Acc_Val", "F1_Val"], ascending=[False, False])

print("Cea mai bună configurație găsită pentru HistGB este:\n")
print(df_istoric_histgb_sorted.iloc[0])

In [ ]:
X_test = df_test.drop(columns=['vacation', 'salary'])
y_test = df_test['vacation']

rezultate_finale = []

X_train_raw = df_simple.drop(columns=['vacation', 'salary'])
Y_train_raw = df_simple['vacation']

le_final = LabelEncoder()
Y_train_enc = le_final.fit_transform(Y_train_raw)
Y_test_enc = le_final.transform(y_test)

X_train_enc = pd.get_dummies(X_train_raw, columns=discret_cols, dummy_na=False)
X_test_enc = pd.get_dummies(X_test, columns=discret_cols, dummy_na=False)
X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

dt_final = DecisionTreeClassifier(criterion='entropy',
                                  min_samples_leaf=150,
                                  max_depth=10)
dt_final.fit(X_train_enc, Y_train_enc)
pred_dt = dt_final.predict(X_test_enc)

rezultate_finale.append({
    'Model': 'DecisionTree',
    'Accuratete': accuracy_score(Y_test_enc, pred_dt),
    'Precizie': precision_score(Y_test_enc, pred_dt, average='weighted', zero_division=0),
    'Recall': recall_score(Y_test_enc, pred_dt, average='weighted', zero_division=0),
    'F1_score': f1_score(Y_test_enc, pred_dt, average='weighted', zero_division=0)
})

rf_final = RandomForestClassifier(
            n_estimators=48,
            max_depth=58,
            min_samples_leaf=4
        )
rf_final.fit(X_train_enc, Y_train_enc)
pred_rf = rf_final.predict(X_test_enc)

rezultate_finale.append({
    'Model': 'RandomForest',
    'Accuratete': accuracy_score(Y_test_enc, pred_rf),
    'Precizie': precision_score(Y_test_enc, pred_rf, average='weighted', zero_division=0),
    'Recall': recall_score(Y_test_enc, pred_rf, average='weighted', zero_division=0),
    'F1_score': f1_score(Y_test_enc, pred_rf, average='weighted', zero_division=0)
})

histgb_final = HistGradientBoostingClassifier(
            max_iter=400,
            max_depth=20,
            learning_rate=0.10,
            max_bins=40
        )
histgb_final.fit(X_train_enc, Y_train_enc)
pred_histgb = histgb_final.predict(X_test_enc)

rezultate_finale.append({
    'Model': 'HistGradientBoosting',
    'Accuratete': accuracy_score(Y_test_enc, pred_histgb),
    'Precizie': precision_score(Y_test_enc, pred_histgb, average='weighted', zero_division=0),
    'Recall': recall_score(Y_test_enc, pred_histgb, average='weighted', zero_division=0),
    'F1_score': f1_score(Y_test_enc, pred_histgb, average='weighted', zero_division=0)
})

df_rezultate_test = pd.DataFrame(rezultate_finale)
print('\nRezultate finale pe setul de test:')
display(df_rezultate_test)


In [ ]:
# Determinare model cel mai bun
best_model_idx = df_rezultate_test['Accuratete'].idxmax()
best_model_name = df_rezultate_test.loc[best_model_idx, 'Model']

if best_model_name == 'DecisionTree':
    best_pred = pred_dt
    best_model = dt_final
elif best_model_name == 'RandomForest':
    best_pred = pred_rf
    best_model = rf_final
else:
    best_pred = pred_histgb
    best_model = histgb_final

cm = confusion_matrix(Y_test_enc, best_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=le_final.classes_, 
            yticklabels=le_final.classes_,
            cbar_kws={'label': 'Count'})
plt.title(f'Confusion Matrix - {best_model_name} (Best Model)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()



In [ ]:
df_comparatie = df_rezultate_test[['Model', 'Accuratete', 'Precizie', 'Recall', 'F1_score']].copy()
df_comparatie.index = np.arange(1, len(df_comparatie) + 1)
df_comparatie.index.name = 'Nr. Algoritm'

def highlight_max_per_col(df, cols):
    styled = df.style.format(precision=4)
    for col in cols:
        styled = styled.apply(
            lambda s: ['font-weight: bold; background-color: #B7E1CD' if v == s.max() else '' for v in s],
            subset=[col]
        )
    return styled

styled = highlight_max_per_col(df_comparatie, ['Accuratete', 'Precizie', 'Recall', 'F1_score'])
max_f1 = df_comparatie['F1_score'].max()
styled = styled.apply(lambda s: ['background-color: #B7E1CD' if v == max_f1 else '' for v in s], subset=['F1_score'])
display(styled)

## Regression models training and evaluation.

In [ ]:
"""REGRESSION - Experimente LinearRegression si Ridge"""

X_reg = df_simple.drop(columns=['vacation', 'salary'], errors='ignore')
y_reg = df_simple['salary']

# Split first to avoid data leakage, then fit OneHotEncoder on train and transform validation
X_train_reg, X_val_reg, y_train_reg, y_val_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

X_train_reg = pd.get_dummies(X_train_reg, columns=discret_cols, dummy_na=False)
X_val_reg = pd.get_dummies(X_val_reg, columns=discret_cols, dummy_na=False)
X_val_reg = X_val_reg.reindex(columns=X_train_reg.columns, fill_value=0)

istorid_lr_ridge = []

def experiment_linear(model_name, model, X_train, y_train, X_val, y_val, table):
    model.fit(X_train, y_train)
    
    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)
    
    mse_train = mean_squared_error(y_train, y_pred_train)
    mse_val = mean_squared_error(y_val, y_pred_val)
    rmse_train = np.sqrt(mse_train)
    rmse_val = np.sqrt(mse_val)
    mae_val = mean_absolute_error(y_val, y_pred_val)
    r2_val = r2_score(y_val, y_pred_val)
    
    table.append({
        'Model': model_name,
        'MSE_Train': mse_train,
        'MSE_Val': mse_val,
        'RMSE_Train': rmse_train,
        'RMSE_Val': rmse_val,
        'MAE_Val': mae_val,
        'R2_Val': r2_val
    })
    return model, y_pred_val

# LinearRegression baseline
lr_model, _ = experiment_linear(
    'LinearRegression', LinearRegression(),
    X_train_reg, y_train_reg, X_val_reg, y_val_reg,
    istorid_lr_ridge
)

# Ridge cu diferiti alphas
alphas_ridge = [10, 15, 20]
for alpha in alphas_ridge:
    ridge_model, _ = experiment_linear(
        f'Ridge(alpha={alpha})', Ridge(alpha=alpha),
        X_train_reg, y_train_reg, X_val_reg, y_val_reg,
        istorid_lr_ridge
    )

df_lr_ridge = pd.DataFrame(istorid_lr_ridge)
df_lr_ridge_sorted = df_lr_ridge.sort_values(by='RMSE_Val', ascending=True)

print('Rezultate LinearRegression si Ridge:')
print(df_lr_ridge_sorted.to_string())
print(f'\nCel mai bun model Ridge/LR: {df_lr_ridge_sorted.iloc[0]["Model"]}')

In [ ]:
istorid_catboost_reg = []

def experiment_catboost_reg(model, dataset_details, X_train, y_train, X_val, y_val, table):
    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        use_best_model=True,
        verbose=0
    )
    
    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)
    
    mse_train = mean_squared_error(y_train, y_pred_train)
    mse_val = mean_squared_error(y_val, y_pred_val)
    rmse_train = np.sqrt(mse_train)
    rmse_val = np.sqrt(mse_val)
    mae_val = mean_absolute_error(y_val, y_pred_val)
    r2_val = r2_score(y_val, y_pred_val)
    
    table.append({
        'Model': f'CatBoost ({dataset_details})',
        'MSE_Train': mse_train,
        'MSE_Val': mse_val,
        'RMSE_Train': rmse_train,
        'RMSE_Val': rmse_val,
        'MAE_Val': mae_val,
        'R2_Val': r2_val
    })
    
    return model, y_pred_val, model.get_evals_result()

model_cb_1 = CatBoostRegressor(
    iterations=500,
    learning_rate=0.15,
    depth=6,
    eval_metric='RMSE',
)
catboost_model_1, _, evals_1 = experiment_catboost_reg(
    model_cb_1, 'iter=500, lr=0.8', X_train_reg, y_train_reg, X_val_reg, y_val_reg, istorid_catboost_reg
)

model_cb_2 = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    l2_leaf_reg=3,
    early_stopping_rounds=30,
    eval_metric='RMSE',
)
catboost_model_2, _, evals_2 = experiment_catboost_reg(
    model_cb_2, 'iter=1000, lr=0.05, depth=0', X_train_reg, y_train_reg, X_val_reg, y_val_reg, istorid_catboost_reg
)

model_cb_3 = CatBoostRegressor(
    iterations=2000,
    learning_rate=0.10,
    l2_leaf_reg=6,
    early_stopping_rounds=30,
    eval_metric='RMSE',
)
catboost_model_3, _, evals_3 = experiment_catboost_reg(
    model_cb_3, 'iter=2000, lr=0.10', X_train_reg, y_train_reg, X_val_reg, y_val_reg, istorid_catboost_reg
)

df_catboost_reg = pd.DataFrame(istorid_catboost_reg)
df_catboost_reg_sorted = df_catboost_reg.sort_values(by='RMSE_Val', ascending=True)
print('\n Rezultate CatBoostRegressor:')
display(df_catboost_reg)
print(f'\n Cel mai bun model CatBoostRegressor: {df_catboost_reg_sorted.iloc[0]["Model"]}')

In [ ]:
X_test_reg = df_test.drop(columns=['vacation', 'salary'])
y_test_reg = df_test['salary']

X_train_all_reg = df_simple.drop(columns=['vacation', 'salary'])
y_train_all_reg = df_simple['salary']

X_train_all_reg_enc = pd.get_dummies(X_train_all_reg, columns=discret_cols, dummy_na=False)
X_test_reg_enc = pd.get_dummies(X_test_reg, columns=discret_cols, dummy_na=False)
X_test_reg_enc = X_test_reg_enc.reindex(columns=X_train_all_reg_enc.columns, fill_value=0)

rezultate_reg_final = []

lr_final = LinearRegression()
lr_final.fit(X_train_all_reg_enc, y_train_all_reg)
y_pred_lr = lr_final.predict(X_test_reg_enc)
rezultate_reg_final.append({
    'Model': 'LinearRegression',
    'RMSE': np.sqrt(mean_squared_error(y_test_reg, y_pred_lr)),
    'MAE': mean_absolute_error(y_test_reg, y_pred_lr),
    'R2': r2_score(y_test_reg, y_pred_lr)
})

best_alpha = df_lr_ridge_sorted.iloc[0]['Model']
if 'alpha=' in best_alpha:
    alpha_val = float(best_alpha.split('alpha=')[1].rstrip(')'))
else:
    alpha_val = 0.1

ridge_final = Ridge(alpha=alpha_val)
ridge_final.fit(X_train_all_reg_enc, y_train_all_reg)
y_pred_ridge = ridge_final.predict(X_test_reg_enc)
rezultate_reg_final.append({
    'Model': f'Ridge(alpha={alpha_val})',
    'RMSE': np.sqrt(mean_squared_error(y_test_reg, y_pred_ridge)),
    'MAE': mean_absolute_error(y_test_reg, y_pred_ridge),
    'R2': r2_score(y_test_reg, y_pred_ridge)
})

catboost_final = CatBoostRegressor(
    iterations=2000,
    learning_rate=0.10,
    l2_leaf_reg=6,
    early_stopping_rounds=30,
    eval_metric='RMSE',
    verbose=0
)

catboost_final.fit(X_train_all_reg_enc, y_train_all_reg)
y_pred_catboost = catboost_final.predict(X_test_reg_enc)
rezultate_reg_final.append({
    'Model': 'CatBoostRegressor ',
    'RMSE': np.sqrt(mean_squared_error(y_test_reg, y_pred_catboost)),
    'MAE': mean_absolute_error(y_test_reg, y_pred_catboost),
    'R2': r2_score(y_test_reg, y_pred_catboost)
})

df_rezultate_reg_test = pd.DataFrame(rezultate_reg_final)
print('\nRezultate finale REGRESIE pe test set:')
display(df_rezultate_reg_test)

In [ ]:
# PLOTS
plt.figure(figsize=(14, 5))

# Plot 1: Evolutia comparativa pe setul de antrenare a mai multor experimente
plt.subplot(1, 2, 1)
plt.plot(evals_1['learn']['RMSE'], label='iter=500, lr=0.05')
plt.plot(evals_2['learn']['RMSE'], label='iter=1000, lr=0.05')
plt.plot(evals_3['learn']['RMSE'], label='iter=2000, lr=0.10')
plt.title('Evoluția comparativă pe setul de antrenare')
plt.xlabel('Iterații (Epochs)')
plt.ylabel('RMSE (Loss)')
plt.legend()
plt.grid(True)

# Plot 2: Comportament comparativ pe setul de antrenare versus cel de validare pentru cel mai bun model
plt.subplot(1, 2, 2)
best_model_name = df_catboost_reg_sorted.iloc[0]['Model']
if '0.10' in best_model_name:
    best_evals = evals_3
elif '1000' in best_model_name:
    best_evals = evals_2
else:
    best_evals = evals_1

plt.plot(best_evals['learn']['RMSE'], label='Train RMSE', color='green')
plt.plot(best_evals['validation']['RMSE'], label='Validation RMSE', color='red')
plt.title(f'Train vs Validation pt cel mai bun model')
plt.xlabel('Iterații (Epochs)')
plt.ylabel('RMSE (Loss)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# KAGGLE VALIDATION

In [ ]:
"""KAGGLE REGRESSION TEST"""
X_test = df_test.drop(columns=['vacation', 'salary'])
X_simple = df_simple.drop(columns=['vacation', 'salary'])

X_final = pd.concat([X_simple, X_test], ignore_index=True)
Y_final = pd.concat([df_simple['salary'], df_test['salary']], ignore_index=True)

X_final_enc = pd.get_dummies(X_final, columns=discret_cols, dummy_na=False)
X_test_kaggle = pd.get_dummies(df_kaggle.copy(), columns=discret_cols, dummy_na=False)
X_test_kaggle = X_test_kaggle.reindex(columns=X_final_enc.columns, fill_value=0)

catboost_final = CatBoostRegressor(
    iterations=2000,
    learning_rate=0.10,
    l2_leaf_reg=6,
    early_stopping_rounds=30,
    eval_metric='RMSE',
    verbose=0
)

catboost_final.fit(X_final_enc, Y_final)
y_pred_catboost = catboost_final.predict(X_test_kaggle)

sub = pd.DataFrame({
    'id': submission_ids,
    'target': y_pred_catboost
})

out_path = 'submission_regression.csv'
sub.to_csv(out_path, index=False)

In [ ]:
"""KAGGLE CLASSIFICATION TEST - HistGradientBoosting"""
X_train_cls = df_simple.drop(columns=['vacation', 'salary'], errors='ignore')
y_train_cls = df_simple['vacation']

X_test_cls = df_test.drop(columns=['vacation', 'salary'], errors='ignore')
y_test_cls = df_test['vacation']

X_final_cls = pd.concat([X_train_cls, X_test_cls], ignore_index=True)
y_final_cls = pd.concat([y_train_cls, y_test_cls], ignore_index=True)

X_final_enc_cls = pd.get_dummies(X_final_cls, columns=discret_cols, dummy_na=False)
X_test_kaggle_cls = pd.get_dummies(df_kaggle.copy(), columns=discret_cols, dummy_na=False)
X_test_kaggle_cls = X_test_kaggle_cls.reindex(columns=X_final_enc_cls.columns, fill_value=0)

le_kaggle = LabelEncoder()
y_final_cls_enc = le_kaggle.fit_transform(y_final_cls)

histgb_kaggle = RandomForestClassifier(
            n_estimators=48,
            max_depth=58,
            min_samples_leaf=4
        )

histgb_kaggle.fit(X_final_enc_cls, y_final_cls_enc)
y_pred_kaggle_enc = histgb_kaggle.predict(X_test_kaggle_cls)
y_pred_kaggle = le_kaggle.inverse_transform(y_pred_kaggle_enc)

sub_histgb = pd.DataFrame({
    'id': submission_ids,
    'target': y_pred_kaggle
})

out_path_histgb = 'submission_random_forest_classification.csv'
sub_histgb.to_csv(out_path_histgb, index=False)